# Wildfire Input Data Generator using Google Earth Engine

This notebook uses **Google Earth Engine (GEE)** to:

- Select **pre-fire** and **post-fire Sentinel-2 imagery**
- Apply **cloud filtering** (`< 10%`)
- Generate a **1 km bounding box** around the fire location
- Export Sentinel-2 imagery bands:
  - **NIR** (`B8`)
  - **SWIR1** (`B11`)
  - **SWIR2** (`B12`)
  - **SCL** (Scene Classification Layer)
- Export terrain layers:
  - **DEM**
  - **Slope**
  - **Aspect**
- Export the **bounding box** as **GeoJSON**

## User Inputs
- Fire latitude
- Fire longitude
- Fire date

## Default behavior
- If no fire perimeter is provided, the notebook:
  - Creates a small **50 m buffer** around the fire coordinate
  - Creates a **1 km buffer**
  - Uses the **bounding box of the 1 km buffer** as the export extent

In [ ]:
# Install required packages in Colab
!pip install -q earthengine-api geemap geopandas

## Import packages
This cell imports the Python libraries needed for:
- Google Earth Engine access
- Google Drive file writing
- JSON/GeoJSON export
- Date handling

In [2]:
# Core Python packages
import os
import json
from datetime import datetime, timedelta

# Google Earth Engine and geemap
import ee
import geemap

## Define output folder in Google Drive
This cell mounts Google Drive and creates an output folder where exported files
such as the bounding box GeoJSON can be saved.

In [4]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define your output folder in Google Drive
OUTPUT_FOLDER = "/content/drive/MyDrive/Wildfire_Input_Data"

# Create the folder if it does not exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Output folder ready: {OUTPUT_FOLDER}")

Mounted at /content/drive
Output folder ready: /content/drive/MyDrive/Wildfire_Input_Data


## Initialize Google Earth Engine
This cell authenticates and initializes Earth Engine in Colab.
Run this once at the beginning of the session.

In [6]:
# Authenticate and initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine is already initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project='class7316')
    print("Earth Engine initialized successfully.")# Trigger the authentication file


Earth Engine initialized successfully.


## Define wildfire input parameters

This section defines the wildfire location, date, and optional fire perimeter.

Workflow:
- If a fire perimeter GeoJSON is provided, it is used as the fire geometry.
- If no fire perimeter is provided, a small buffer is created around the fire point.
- A larger bounding box is then created from the fire geometry.
- That bounding box is used as the processing and export extent for all imagery and terrain layers.

In [13]:
# ==============================
# USER INPUTS
# ==============================

# Fire location
fire_lat = 34.1234
fire_lon = -117.5678

# Fire date (YYYY-MM-DD)
fire_date = "2024-08-10"

# Optional fire perimeter GeoJSON path
# Set to None if no fire perimeter is available
fire_perimeter_path = None
# Example:
# fire_perimeter_path = "/content/drive/MyDrive/Wildfire_Input_Data/fire_perimeter.geojson"

# Default point buffer (meters) if no fire perimeter is available
# This creates a small substitute fire area around the point
default_point_buffer_m = 50

# Bounding area buffer (meters) around the fire geometry
# This is used to create the final processing/export bounding box
bbox_buffer_m = 1000

# Cloud threshold for Sentinel-2 filtering
max_cloud = 10

## Create fire geometry and bounding box

This cell:
- creates the fire point from latitude and longitude
- uses the fire perimeter if provided
- otherwise uses a small buffer around the fire point
- creates a bounding box around the fire geometry

The final bounding box is used as the processing boundary for imagery and terrain products.

In [14]:
# Create fire point from latitude and longitude
fire_point = ee.Geometry.Point([fire_lon, fire_lat])

# --------------------------------------------------
# Create fire geometry
# --------------------------------------------------
if fire_perimeter_path is not None:
    # Read GeoJSON fire perimeter from file
    with open(fire_perimeter_path, "r") as f:
        fire_geojson = json.load(f)

    # Use the first feature geometry from the GeoJSON
    fire_geom = ee.Geometry(fire_geojson["features"][0]["geometry"])
    print("Using fire perimeter from GeoJSON.")

else:
    # If no fire perimeter is available, use a small buffer around the fire point
    fire_geom = fire_point.buffer(default_point_buffer_m)
    print(f"No fire perimeter provided. Using {default_point_buffer_m} m buffer around fire point.")

# --------------------------------------------------
# Create bounding box from fire geometry
# --------------------------------------------------
bbox_geom = fire_geom.buffer(bbox_buffer_m).bounds()

print("Fire geometry created.")
print("Bounding box created and will be used for imagery, terrain layers, and exports.")

No fire perimeter provided. Using 50 m buffer around fire point.
Fire geometry created.
Bounding box created and will be used for imagery, terrain layers, and exports.


## Find the nearest pre-fire and post-fire Sentinel-2 images

This section identifies the **three nearest cloud-filtered Sentinel-2 images** before and after the fire date.

Workflow:
- filter Sentinel-2 imagery using the **bounding box**
- apply a cloud filter (`CLOUDY_PIXEL_PERCENTAGE < 10`)
- restrict the search to a **±60-day window around the fire date**
- split imagery into:
  - **pre-fire images** (before fire date)
  - **post-fire images** (on or after fire date)
- sort images by acquisition date:
  - pre-fire: nearest date before fire first
  - post-fire: nearest date after fire first
- select the **three closest images** on each side of the fire event

These images are then visualized individually with their acquisition dates, allowing the user to inspect them and choose which dates to export.

In [40]:
# Sentinel-2 Surface Reflectance Harmonized collection
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

# Required bands for wildfire work
required_bands = ["B8", "B12", "SCL"]

# Convert fire date to Earth Engine date
fire_dt_ee = ee.Date(fire_date)

# Define search window around fire date
pre_window_start = fire_dt_ee.advance(-60, "day")
pre_window_end = fire_dt_ee

post_window_start = fire_dt_ee
post_window_end = fire_dt_ee.advance(60, "day")

# -----------------------------------------
# Pre-fire collection
# Sorted descending so the nearest date before fire comes first
# -----------------------------------------
pre_collection = (
    s2.filterBounds(bbox_geom)
      .filterDate(pre_window_start, pre_window_end)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
      .select(required_bands)
      .sort("system:time_start", False)
)

# -----------------------------------------
# Post-fire collection
# Sorted ascending so the nearest date after fire comes first
# -----------------------------------------
post_collection = (
    s2.filterBounds(bbox_geom)
      .filterDate(post_window_start, post_window_end)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
      .select(required_bands)
      .sort("system:time_start", True)
)

# Get the nearest 3 images from each side
pre_list = pre_collection.limit(3).toList(3)
post_list = post_collection.limit(3).toList(3)

print("Nearest pre-fire and post-fire image lists created.")

Nearest pre-fire and post-fire image lists created.


## Extract and print the available image dates

This section prints the acquisition dates of the three nearest pre-fire and three nearest post-fire images.

The user will use these dates to decide which image to export later.

In [41]:
# Helper function to extract image dates from an Earth Engine list
def get_image_dates(image_list, n=3):
    dates = []
    for i in range(n):
        img = ee.Image(image_list.get(i))
        date_str = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo()
        dates.append(date_str)
    return dates

# Extract dates
pre_dates = get_image_dates(pre_list, 3)
post_dates = get_image_dates(post_list, 3)

print("Nearest Pre-fire Dates:")
for i, d in enumerate(pre_dates, start=1):
    print(f"{i}. {d}")

print("\nNearest Post-fire Dates:")
for i, d in enumerate(post_dates, start=1):
    print(f"{i}. {d}")

Nearest Pre-fire Dates:
1. 2024-08-05
2. 2024-07-31
3. 2024-07-26

Nearest Post-fire Dates:
1. 2024-08-10
2. 2024-08-15
3. 2024-08-20


## Visualize the nearest pre-fire and post-fire images

This section adds the three nearest pre-fire and three nearest post-fire images to the map.

Each image is labeled using its acquisition date.

Displayed layers:
- **B8** = NIR
- **B12** = SWIR2

The map also includes:
- the **fire geometry**
- the **bounding box**

This helps the user visually inspect the candidate images before choosing export dates.

In [42]:
# Create interactive map
Map = geemap.Map(center=[fire_lat, fire_lon], zoom=13)

# Visualization parameters for wildfire interpretation
vis_false_color = {
    "bands": ["B12", "B8"],
    "min": 0,
    "max": 4000
}

# -----------------------------------------
# Add nearest pre-fire images
# -----------------------------------------
for i in range(3):
    img = ee.Image(pre_list.get(i)).select(required_bands).clip(bbox_geom)
    date_str = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo()
    Map.addLayer(img, vis_false_color, f"Pre-fire {date_str}")

# -----------------------------------------
# Add nearest post-fire images
# -----------------------------------------
for i in range(3):
    img = ee.Image(post_list.get(i)).select(required_bands).clip(bbox_geom)
    date_str = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo()
    Map.addLayer(img, vis_false_color, f"Post-fire {date_str}")

# Add fire geometry and bounding box
Map.addLayer(fire_geom, {"color": "yellow"}, "Fire Geometry")
Map.addLayer(bbox_geom, {"color": "red"}, "Bounding Box")
Map.addLayer(fire_point, {"color": "blue"}, "Fire Point")

Map

Map(center=[34.1234, -117.5678], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

## Select the pre-fire and post-fire dates to export

After reviewing the candidate images, choose one pre-fire date and one post-fire date from the printed lists.

These selected dates will be used to retrieve and export the final Sentinel-2 images.

In [43]:
# ==============================
# USER-SELECTED EXPORT DATES
# ==============================

# Choose one date from the printed pre-fire list
selected_pre_date = pre_dates[0]

# Choose one date from the printed post-fire list
selected_post_date = post_dates[1]

print("Selected pre-fire date :", selected_pre_date)
print("Selected post-fire date:", selected_post_date)

Selected pre-fire date : 2024-08-05
Selected post-fire date: 2024-08-15


## Load the selected Sentinel-2 images for export

This section retrieves the user-selected pre-fire and post-fire images based on the chosen dates.

The selected images are clipped to the bounding box and include:
- **B8** = NIR
- **B12** = SWIR2
- **SCL** = Scene Classification Layer

In [44]:
# Helper function to get one Sentinel-2 image by exact date
def get_image_by_date(collection, date_string, region, bands):
    start = ee.Date(date_string)
    end = start.advance(1, "day")

    image = (
        collection.filterBounds(region)
                  .filterDate(start, end)
                  .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud))
                  .select(bands)
                  .first()
    )
    return ee.Image(image).clip(region)

# Load selected pre-fire and post-fire images
selected_pre_image = get_image_by_date(s2, selected_pre_date, bbox_geom, required_bands)
selected_post_image = get_image_by_date(s2, selected_post_date, bbox_geom, required_bands)

print("Selected Sentinel-2 images loaded and clipped to bounding box.")

Selected Sentinel-2 images loaded and clipped to bounding box.


## Visualize the final selected pre-fire and post-fire images

This section displays the two images chosen by the user for export.

In [45]:
Map = geemap.Map(center=[fire_lat, fire_lon], zoom=13)

vis_false_color = {
    "bands": ["B12", "B8"],
    "min": 0,
    "max": 4000
}

Map.addLayer(selected_pre_image, vis_false_color, f"Selected Pre-fire {selected_pre_date}")
Map.addLayer(selected_post_image, vis_false_color, f"Selected Post-fire {selected_post_date}")
Map.addLayer(fire_geom, {"color": "yellow"}, "Fire Geometry")
Map.addLayer(bbox_geom, {"color": "red"}, "Bounding Box")

Map

Map(center=[34.1234, -117.5678], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

## Load DEM, slope, and aspect

This section loads the DEM and derives terrain layers using the same bounding box extent.
These layers do not need date selection.

In [34]:
# Load DEM
dem = ee.Image("USGS/SRTMGL1_003").clip(bbox_geom)

# Derive terrain products
terrain = ee.Terrain.products(dem)
slope = terrain.select("slope").clip(bbox_geom)
aspect = terrain.select("aspect").clip(bbox_geom)

print("DEM, slope, and aspect created.")

DEM, slope, and aspect created.


## Export selected imagery, terrain layers, and preprocessing outputs

This section exports the final wildfire input data to Google Drive using the bounding box as the common spatial extent.

### Sentinel-2 exports
For the selected pre-fire and post-fire dates, the notebook exports:

- **B8 (NIR)** and **B12 (SWIR2)** together as spectral imagery
- **SCL (Scene Classification Layer)** separately as a classification mask

The SCL layer is exported separately because it has a different data type from the spectral bands.

### Terrain exports
The notebook also exports:
- **DEM**
- **Slope**
- **Aspect**

### Instruction
1. Confirm the selected pre-fire and post-fire dates.
2. Run the export cell.
3. Check task status after export.
4. If a task fails, print the task status to inspect the error message.

In [54]:
# Google Drive folder for Earth Engine exports
GEE_DRIVE_FOLDER = "Wildfire_Input_Data"

# Export scales
export_scale_s2 = 20
export_scale_dem = 30

# -----------------------------------------
# Export selected pre-fire spectral bands
# -----------------------------------------
task_pre = ee.batch.Export.image.toDrive(
    image=selected_pre_image.select(["B8", "B12"]).toUint16(),
    description="selected_pre_fire_s2_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix=f"pre_fire_{selected_pre_date}_B8_B12",
    region=bbox_geom,
    scale=export_scale_s2,
    maxPixels=1e13
)
task_pre.start()

# -----------------------------------------
# Export selected post-fire spectral bands
# -----------------------------------------
task_post = ee.batch.Export.image.toDrive(
    image=selected_post_image.select(["B8", "B12"]).toUint16(),
    description="selected_post_fire_s2_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix=f"post_fire_{selected_post_date}_B8_B12",
    region=bbox_geom,
    scale=export_scale_s2,
    maxPixels=1e13
)
task_post.start()

# -----------------------------------------
# Export pre-fire SCL separately
# -----------------------------------------
task_pre_scl = ee.batch.Export.image.toDrive(
    image=selected_pre_image.select("SCL").toByte(),
    description="selected_pre_fire_scl_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix=f"pre_fire_{selected_pre_date}_SCL",
    region=bbox_geom,
    scale=20,
    maxPixels=1e13
)
task_pre_scl.start()

# -----------------------------------------
# Export post-fire SCL separately
# -----------------------------------------
task_post_scl = ee.batch.Export.image.toDrive(
    image=selected_post_image.select("SCL").toByte(),
    description="selected_post_fire_scl_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix=f"post_fire_{selected_post_date}_SCL",
    region=bbox_geom,
    scale=20,
    maxPixels=1e13
)
task_post_scl.start()

# -----------------------------------------
# Export DEM
# -----------------------------------------
task_dem = ee.batch.Export.image.toDrive(
    image=dem,
    description="dem_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix="dem_srtm",
    region=bbox_geom,
    scale=export_scale_dem,
    maxPixels=1e13
)
task_dem.start()

# -----------------------------------------
# Export slope
# -----------------------------------------
task_slope = ee.batch.Export.image.toDrive(
    image=slope,
    description="slope_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix="slope_srtm",
    region=bbox_geom,
    scale=export_scale_dem,
    maxPixels=1e13
)
task_slope.start()

# -----------------------------------------
# Export aspect
# -----------------------------------------
task_aspect = ee.batch.Export.image.toDrive(
    image=aspect,
    description="aspect_export",
    folder=GEE_DRIVE_FOLDER,
    fileNamePrefix="aspect_srtm",
    region=bbox_geom,
    scale=export_scale_dem,
    maxPixels=1e13
)
task_aspect.start()

print("Export tasks started.")

Export tasks started.


## Export task status

After running the export cell, each task will show a status.

### What the status means

- **READY** → The task has been created and queued, but has not started yet.
- **RUNNING** → The export is currently in progress.
- **COMPLETED** → The export finished successfully and the file is available in Google Drive.
- **FAILED** → The export did not complete due to an error. Check the error message for details.

### What to do

- If the status is **READY**, wait a few moments — tasks will start automatically.
- If the status is **RUNNING**, allow time for processing (larger areas take longer).
- If the status is **COMPLETED**, you can proceed to download or use the exported data.
- If the status is **FAILED**, inspect the task details and rerun after fixing the issue.

You can re-run the status check cell to monitor progress.

In [70]:
print("Pre-fire export:", task_pre.status()['state'])
print("Post-fire export:", task_post.status()['state'])
print("DEM export:", task_dem.status()['state'])
print("Slope export:", task_slope.status()['state'])
print("Aspect export:", task_aspect.status()['state'])

Pre-fire export: COMPLETED
Post-fire export: COMPLETED
DEM export: COMPLETED
Slope export: COMPLETED
Aspect export: COMPLETED


## Export selected imagery, terrain layers, and preprocessing outputs

This section exports the final wildfire input data to the specified **output folder** in Google Drive.

### Files exported
The following datasets are exported using the **bounding box** as the common spatial extent:

#### Sentinel-2 imagery
For the user-selected pre-fire and post-fire dates, the notebook exports:

- **B8 (NIR)** — resampled to **20 m**
- **B12 (SWIR2)** — native **20 m**

These two bands are exported together as spectral imagery and are used for wildfire analysis, especially for calculating indices such as **NBR** and **dNBR**.

- **SCL (Scene Classification Layer)** — **20 m**

The SCL layer is exported **separately** as a classification mask. It can be used for:
- cloud masking
- shadow masking
- snow masking
- general data cleaning before analysis

#### Terrain layers
The notebook also exports:
- **DEM**
- **Slope**
- **Aspect**

These terrain layers are clipped to the same bounding box so they align spatially with the Sentinel-2 imagery.

#### Bounding box
The final analysis and export boundary is also saved as a **GeoJSON** file in the output folder.

---

### Preprocessing
Before export:
- the fire geometry is defined using either:
  - the provided fire perimeter GeoJSON, or
  - a fallback buffered fire point
- a **bounding box** is created from that fire geometry
- Sentinel-2 images are filtered by:
  - the bounding box
  - a ±60-day window around the fire date
  - the cloud threshold
- the user inspects the nearest pre-fire and post-fire candidate images
- one pre-fire image and one post-fire image are selected for export

---

### Export location
- Local files (e.g., bounding box GeoJSON) are written to the notebook **output folder** in Google Drive.
- Raster exports from Earth Engine are saved in the specified **Google Drive export folder**.

---

### Instructions
Before running the export cell:

1. Confirm that the correct pre-fire and post-fire images have been selected.
2. Confirm that the output folder path is correct.
3. Run the export cell to send the imagery and terrain layers to Google Drive.
4. Monitor the task status:
   - **READY** → queued
   - **RUNNING** → processing
   - **COMPLETED** → finished successfully
   - **FAILED** → check error message
5. Once completed, verify that all exported files are available in Google Drive.

In [67]:

# Make sure the local Google Drive output folder exists
OUTPUT_FOLDER = "/content/drive/MyDrive/Wildfire_Input_Data"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# -----------------------------------------
# Create a 500 m circular fire perimeter
# -----------------------------------------
fire_perimeter_geom = fire_point.buffer(500)

# Convert Earth Engine geometry to GeoJSON-style dictionary
fire_perimeter_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "name": "fire_perimeter",
                "fire_date": fire_date,
                "latitude": fire_lat,
                "longitude": fire_lon,
                "buffer_m": 500
            },
            "geometry": fire_perimeter_geom.getInfo()
        }
    ]
}

# Save as GeoJSON in the same Drive folder
fire_perimeter_path = os.path.join(OUTPUT_FOLDER, "fire_perimeter.geojson")

with open(fire_perimeter_path, "w") as f:
    json.dump(fire_perimeter_geojson, f, indent=2)

print(f"Fire perimeter GeoJSON saved to: {fire_perimeter_path}")

Fire perimeter GeoJSON saved to: /content/drive/MyDrive/Wildfire_Input_Data/fire_perimeter.geojson
